**Name:** Rahul Raj &nbsp;&nbsp;|&nbsp;&nbsp; **Roll No.:** 12341680 &nbsp;&nbsp;|&nbsp;&nbsp; **Lab Assignment 6** — Building and Comparing Language Models

# Assignment: Building and Comparing Language Models

**Objective:** In this assignment, you have to implement RNN language models, evaluate their performance using the perplexity metric, and compare their result with Neural language model of fixed window size and Probabilistic language Model.

1.  **A Neural RNN-based Language Model** - A modern deep learning approach.
---

## What is Perplexity?

Perplexity is the standard evaluation metric for language models. It measures how well a probability model predicts a sample. A lower perplexity indicates that the model is better at predicting the text.

It is calculated as the exponentiated average negative log-likelihood of a sequence of words $W = (w_1, w_2, ..., w_N)$:

$$ Perplexity(W) = P(w_1, w_2, ..., w_N)^{-\frac{1}{N}} = \sqrt[N]{\frac{1}{P(w_1, w_2, ..., w_N)}} $$

For our n-gram and RNN models, we will compute this on a held-out test set.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import collections
import re

## Data Preparation

We have to use below corpus for this assignment for training . We'll also split it into a training set and a testing set.

In [2]:
corpus = """
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in a villa on the Riviera. (Though I rather thought it would have been Rome or Florence.)

"The height of his glory"--that was what the women called it. I can hear Mrs. Gideon Thwing--his last Chicago sitter--deploring his unaccountable abdication. "Of course it's going to send the value of my picture 'way up; but I don't think of that, Mr. Rickham--the loss to Arrt is all I think of." The word, on Mrs. Thwing's lips, multiplied its _rs_ as though they were reflected in an endless vista of mirrors. And it was not only the Mrs. Thwings who mourned. Had not the exquisite Hermia Croft, at the last Grafton Gallery show, stopped me before Gisburn's "Moon-dancers" to say, with tears in her eyes: "We shall not look upon its like again"?

Well!--even through the prism of Hermia's tears I felt able to face the fact with equanimity. Poor Jack Gisburn! The women had made him--it was fitting that they should mourn him. Among his own sex fewer regrets were heard, and in his own trade hardly a murmur. Professional jealousy? Perhaps. If it were, the honour of the craft was vindicated by little Claude Nutley, who, in all good faith, brought out in the Burlington a very handsome "obituary" on Jack--one of those showy articles stocked with random technicalities that I have heard (I won't say by whom) compared to Gisburn's painting. And so--his resolve being apparently irrevocable--the discussion gradually died out, and, as Mrs. Thwing had predicted, the price of "Gisburns" went up.

It was not till three years later that, in the course of a few weeks' idling on the Riviera, it suddenly occurred to me to wonder why Gisburn had given up his painting. On reflection, it really was a tempting problem. To accuse his wife would have been too easy--his fair sitters had been denied the solace of saying that Mrs. Gisburn had "dragged him down." For Mrs. Gisburn--as such--had not existed till nearly a year after Jack's resolve had been taken. It might be that he had married her--since he liked his ease--because he didn't want to go on painting; but it would have been hard to prove that he had given up his painting because he had married her.

Of course, if she had not dragged him down, she had equally, as Miss Croft contended, failed to "lift him up"--she had not led him back to the easel. To put the brush into his hand again--what a vocation for a wife! But Mrs. Gisburn appeared to have disdained it--and I felt it might be interesting to find out why.

The desultory life of the Riviera lends itself to such purely academic speculations; and having, on my way to Monte Carlo, caught a glimpse of Jack's balustraded terraces between the pines, I had myself borne thither the next day.

I found the couple at tea beneath their palm-trees; and Mrs. Gisburn's welcome was so genial that, in the ensuing weeks, I claimed it frequently. It was not that my hostess was "interesting": on that point I could have given Miss Croft the fullest reassurance. It was just because she was _not_ interesting--if I may be pardoned the bull--that I found her so. For Jack, all his life, had been surrounded by interesting women: they had fostered his art, it had been reared in the hot-house of their adulation. And it was therefore instructive to note what effect the "deadening atmosphere of mediocrity" (I quote Miss Croft) was having on him.

I have mentioned that Mrs. Gisburn was rich; and it was immediately perceptible that her husband was extracting from this circumstance a delicate but substantial satisfaction. It is, as a rule, the people who scorn money who get most out of it; and Jack's elegant disdain of his wife's big balance enabled him, with an appearance of perfect good-breeding, to transmute it into objects of art and luxury. To the latter, I must add, he remained relatively indifferent; but he was buying Renaissance bronzes and eighteenth-century pictures with a discrimination that bespoke the amplest resources.

"Money's only excuse is to put beauty into circulation," was one of the axioms he laid down across the Sevres and silver of an exquisitely appointed luncheon-table, when, on a later day, I had again run over from Monte Carlo; and Mrs. Gisburn, beaming on him, added for my enlightenment: "Jack is so morbidly sensitive to every form of beauty."

Poor Jack! It had always been his fate to have women say such things of him: the fact should be set down in extenuation. What struck me now was that, for the first time, he resented the tone. I had seen him, so often, basking under similar tributes--was it the conjugal note that robbed them of their savour? No--for, oddly enough, it became apparent that he was fond of Mrs. Gisburn--fond enough not to see her absurdity. It was his own absurdity he seemed to be wincing under--his own attitude as an object for garlands and incense.

"My dear, since I've chucked painting people don't say that stuff about me--they say it about Victor Grindle," was his only protest, as he rose from the table and strolled out onto the sunlit terrace.

I glanced after him, struck by his last word. Victor Grindle was, in fact, becoming the man of the moment--as Jack himself, one might put it, had been the man of the hour. The younger artist was said to have formed himself at my friend's feet, and I wondered if a tinge of jealousy underlay the latter's mysterious abdication. But no--for it was not till after that event that the _rose Dubarry_ drawing-rooms had begun to display their "Grindles."

I turned to Mrs. Gisburn, who had lingered to give a lump of sugar to her spaniel in the dining-room.

"Why _has_ he chucked painting?" I asked abruptly.

She raised her eyebrows with a hint of good-humoured surprise.

"Oh, he doesn't _have_ to now, you know; and I want him to enjoy himself," she said quite simply.

I looked about the spacious white-panelled room, with its _famille-verte_ vases repeating the tones of the pale damask curtains, and its eighteenth-century pastels in delicate faded frames.

"Has he chucked his pictures too? I haven't seen a single one in the house."

A slight shade of constraint crossed Mrs. Gisburn's open countenance. "It's his ridiculous modesty, you know. He says they're not fit to have about; he's sent them all away except one--my portrait--and that I have to keep upstairs."

His ridiculous modesty--Jack's modesty about his pictures? My curiosity was growing like the bean-stalk. I said persuasively to my hostess: "I must really see your portrait, you know."

She glanced out almost timorously at the terrace where her husband, lounging in a hooded chair, had lit a cigar and drawn the Russian deerhound's head between his knees.

"Well, come while he's not looking," she said, with a laugh that tried to hide her nervousness; and I followed her between the marble Emperors of the hall, and up the wide stairs with terra-cotta nymphs poised among flowers at each landing.

In the dimmest corner of her boudoir, amid a profusion of delicate and distinguished objects, hung one of the familiar oval canvases, in the inevitable garlanded frame. The mere outline of the frame called up all Gisburn's past!

Mrs. Gisburn drew back the window-curtains, moved aside a _jardiniere_ full of pink azaleas, pushed an arm-chair away, and said: "If you stand here you can just manage to see it. I had it over the mantel-piece, but he wouldn't let it stay."

Yes--I could just manage to see it--the first portrait of Jack's I had ever had to strain my eyes over! Usually they had the place of honour--say the central panel in a pale yellow or _rose Dubarry_ drawing-room, or a monumental easel placed so that it took the light through curtains of old Venetian point. The more modest place became the picture better; yet, as my eyes grew accustomed to the half-light, all the characteristic qualities came out--all the hesitations disguised as audacities, the tricks of prestidigitation by which, with such consummate skill, he managed to divert attention from the real business of the picture to some pretty irrelevance of detail. Mrs. Gisburn, presenting a neutral surface to work on--forming, as it were, so inevitably the background of her own picture--had lent herself in an unusual degree to the display of this false virtuosity. The picture was one of Jack's "strongest," as his admirers would have put it--it represented, on his part, a swelling of muscles, a congesting of veins, a balancing, straddling and straining, that reminded one of the circus-clown's ironic efforts to lift a feather. It met, in short, at every point the demand of lovely woman to be painted "strongly" because she was tired of being painted "sweetly"--and yet not to lose an atom of the sweetness.

"It's the last he painted, you know," Mrs. Gisburn said with pardonable pride. "The last but one," she corrected herself--"but the other doesn't count, because he destroyed it."

"Destroyed it?" I was about to follow up this clue when I heard a footstep and saw Jack himself on the threshold.

As he stood there, his hands in the pockets of his velveteen coat, the thin brown waves of hair pushed back from his white forehead, his lean sunburnt cheeks furrowed by a smile that lifted the tips of a self-confident moustache, I felt to what a degree he had the same quality as his pictures--the quality of looking cleverer than he was.

His wife glanced at him deprecatingly, but his eyes travelled past her to the portrait.

"Mr. Rickham wanted to see it," she began, as if excusing herself. He shrugged his shoulders, still smiling.

"Oh, Rickham found me out long ago," he said lightly; then, passing his arm through mine: "Come and see the rest of the house."

He showed it to me with a kind of naive suburban pride: the bath-rooms, the speaking-tubes, the dress-closets, the trouser-presses--all the complex simplifications of the millionaire's domestic economy. And whenever my wonder paid the expected tribute he said, throwing out his chest a little: "Yes, I really don't see how people manage to live without that."

Well--it was just the end one might have foreseen for him. Only he was, through it all and in spite of it all--as he had been through, and in spite of, his pictures--so handsome, so charming, so disarming, that one longed to cry out: "Be dissatisfied with your leisure!" as once one had longed to say: "Be dissatisfied with your work!"

But, with the cry on my lips, my diagnosis suffered an unexpected check.

"This is my own lair," he said, leading me into a dark plain room at the end of the florid vista. It was square and brown and leathery: no "effects"; no bric-a-brac, none of the air of posing for reproduction in a picture weekly--above all, no least sign of ever having been used as a studio.

The fact brought home to me the absolute finality of Jack's break with his old life.

"Don't you ever dabble with paint any more?" I asked, still looking about for a trace of such activity.

"Never," he said briefly.

"Or water-colour--or etching?"

His confident eyes grew dim, and his cheeks paled a little under their handsome sunburn.

"Never think of it, my dear fellow--any more than if I'd never touched a brush."

And his tone told me in a flash that he never thought of anything else.

I moved away, instinctively embarrassed by my unexpected discovery; and as I turned, my eye fell on a small picture above the mantel-piece--the only object breaking the plain oak panelling of the room.

"Oh, by Jove!" I said.

It was a sketch of a donkey--an old tired donkey, standing in the rain under a wall.

"By Jove--a Stroud!" I cried.

He was silent; but I felt him close behind me, breathing a little quickly.

"What a wonder! Made with a dozen lines--but on everlasting foundations. You lucky chap, where did you get it?"

He answered slowly: "Mrs. Stroud gave it to me."

"Ah--I didn't know you even knew the Strouds. He was such an inflexible hermit."

"I didn't--till after. . . . She sent for me to paint him when he was dead."

"When he was dead? You?"

I must have let a little too much amazement escape through my surprise, for he answered with a deprecating laugh: "Yes--she's an awful simpleton, you know, Mrs. Stroud. Her only idea was to have him done by a fashionable painter--ah, poor Stroud! She thought it the surest way of proclaiming his greatness--of forcing it on a purblind public. And at the moment I was _the_ fashionable painter."

"Ah, poor Stroud--as you say. Was _that_ his history?"

"That was his history. She believed in him, gloried in him--or thought she did. But she couldn't bear not to have all the drawing-rooms with her. She couldn't bear the fact that, on varnishing days, one could always get near enough to see his pictures. Poor woman! She's just a fragment groping for other fragments. Stroud is the only whole I ever knew."

"You ever knew? But you just said--"

Gisburn had a curious smile in his eyes.

"Oh, I knew him, and he knew me--only it happened after he was dead."

I dropped my voice instinctively. "When she sent for you?"

"Yes--quite insensible to the irony. She wanted him vindicated--and by me!"

He laughed again, and threw back his head to look up at the sketch of the donkey. "There were days when I couldn't look at that thing--couldn't face it. But I forced myself to put it here; and now it's cured me--cured me. That's the reason why I don't dabble any more, my dear Rickham; or rather Stroud himself is the reason."

For the first time my idle curiosity about my companion turned into a serious desire to understand him better.

"I wish you'd tell me how it happened," I said.

He stood looking up at the sketch, and twirling between his fingers a cigarette he had forgotten to light. Suddenly he turned toward me.

"I'd rather like to tell you--because I've always suspected you of loathing my work."

I made a deprecating gesture, which he negatived with a good-humoured shrug.

"Oh, I didn't care a straw when I believed in myself--and now it's an added tie between us!"

He laughed slightly, without bitterness, and pushed one of the deep arm-chairs forward. "There: make yourself comfortable--and here are the cigars you like."

He placed them at my elbow and continued to wander up and down the room, stopping now and then beneath the picture.

"How it happened? I can tell you in five minutes--and it didn't take much longer to happen. . . . I can remember now how surprised and pleased I was when I got Mrs. Stroud's note. Of course, deep down, I had always _felt_ there was no one like him--only I had gone with the stream, echoed the usual platitudes about him, till I half got to think he was a failure, one of the kind that are left behind. By Jove, and he _was_ left behind--because he had come to stay! The rest of us had to let ourselves be swept along or go under, but he was high above the current--on everlasting foundations, as you say.

"Well, I went off to the house in my most egregious mood--rather moved, Lord forgive me, at the pathos of poor Stroud's career of failure being crowned by the glory of my painting him! Of course I meant to do the picture for nothing--I told Mrs. Stroud so when she began to stammer something about her poverty. I remember getting off a prodigious phrase about the honour being _mine_--oh, I was princely, my dear Rickham! I was posing to myself like one of my own sitters.

"Then I was taken up and left alone with him. I had sent all my traps in advance, and I had only to set up the easel and get to work. He had been dead only twenty-four hours, and he died suddenly, of heart disease, so that there had been no preliminary work of destruction--his face was clear and untouched. I had met him once or twice, years before, and thought him insignificant and dingy. Now I saw that he was superb.

"I was glad at first, with a merely aesthetic satisfaction: glad to have my hand on such a 'subject.' Then his strange life-likeness began to affect me queerly--as I blocked the head in I felt as if he were watching me do it. The sensation was followed by the thought: if he _were_ watching me, what would he say to my way of working? My strokes began to go a little wild--I felt nervous and uncertain.

"Once, when I looked up, I seemed to see a smile behind his close grayish beard--as if he had the secret, and were amusing himself by holding it back from me. That exasperated me still more. The secret? Why, I had a secret worth twenty of his! I dashed at the canvas furiously, and tried some of my bravura tricks. But they failed me, they crumbled. I saw that he wasn't watching the showy bits--I couldn't distract his attention; he just kept his eyes on the hard passages between. Those were the ones I had always shirked, or covered up with some lying paint. And how he saw through my lies!

"I looked up again, and caught sight of that sketch of the donkey hanging on the wall near his bed. His wife told me afterward it was the last thing he had done--just a note taken with a shaking hand, when he was down in Devonshire recovering from a previous heart attack. Just a note! But it tells his whole history. There are years of patient scornful persistence in every line. A man who had swum with the current could never have learned that mighty up-stream stroke. . . .

"I turned back to my work, and went on groping and muddling; then I looked at the donkey again. I saw that, when Stroud laid in the first stroke, he knew just what the end would be. He had possessed his subject, absorbed it, recreated it. When had I done that with any of my things? They hadn't been born of me--I had just adopted them. . . .

"Hang it, Rickham, with that face watching me I couldn't do another stroke. The plain truth was, I didn't know where to put it--_I had never known_. Only, with my sitters and my public, a showy splash of colour covered up the fact--I just threw paint into their faces. . . . Well, paint was the one medium those dead eyes could see through--see straight to the tottering foundations underneath. Don't you know how, in talking a foreign language, even fluently, one says half the time not what one wants to but what one can? Well--that was the way I painted; and as he lay there and watched me, the thing they called my 'technique' collapsed like a house of cards. He didn't sneer, you understand, poor Stroud--he just lay there quietly watching, and on his lips, through the gray beard, I seemed to hear the question: 'Are you sure you know where you're coming out?'

"If I could have painted that face, with that question on it, I should have done a great thing. The next greatest thing was to see that I couldn't--and that grace was given me. But, oh, at that minute, Rickham, was there anything on earth I wouldn't have given to have Stroud alive before me, and to hear him say: 'It's not too late--I'll show you how'?

"It _was_ too late--it would have been, even if he'd been alive. I packed up my traps, and went down and told Mrs. Stroud. Of course I didn't tell her _that_--it would have been Greek to her. I simply said I couldn't paint him, that I was too moved. She rather liked the idea--she's so romantic! It was that that made her give me the donkey. But she was terribly upset at not getting the portrait--she did so want him 'done' by some one showy! At first I was afraid she wouldn't let me off--and at my wits' end I suggested Grindle. Yes, it was I who started Grindle: I told Mrs. Stroud he was the 'coming' man, and she told somebody else, and so it got to be true. . . . And he painted Stroud without wincing; and she hung the picture among her husband's things. . . ."

He flung himself down in the arm-chair near mine, laid back his head, and clasping his arms beneath it, looked up at the picture above the chimney-piece.

"I like to fancy that Stroud himself would have given it to me, if he'd been able to say what he thought that day."

And, in answer to a question I put half-mechanically--"Begin again?" he flashed out. "When the one thing that brings me anywhere near him is that I knew enough to leave off?"

He stood up and laid his hand on my shoulder with a laugh. "Only the irony of it is that I _am_ still painting--since Grindle's doing it for me! The Strouds stand alone, and happen once--but there's no exterminating our kind of art."
"""

# Part 1: Neural RNN Language Model

Now, you have to build an RNN to perform the same task. The RNN will process a sequence of words and predict the next one.

### Step 1.1: Create Sequential Data

Unlike the n-gram model, which looks at pairs of words, the RNN can look at a longer history. We need to create input sequences and corresponding targets.

In [3]:
# ============================================================
# Step 1.1: Create Sequential Data
# ============================================================

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


def tokenize(text):
    """Lower-case the text and split it into word / punctuation tokens."""
    text = text.lower().replace("--", " ").replace("_", "")
    # a word (optionally with an internal apostrophe or hyphen) OR a punctuation mark
    return re.findall(r"[a-z]+(?:['-][a-z]+)*|[.,!?;:\"]", text)


tokens = tokenize(corpus)
print(f"Total tokens in the corpus : {len(tokens)}")
print(f"First 20 tokens           : {tokens[:20]}")

# ------------------------------------------------------------
# Train / test split: the first 80% of the text is used for training,
# the last 20% is held out as an unseen test set.
# ------------------------------------------------------------
split_idx = int(0.8 * len(tokens))
train_tokens = tokens[:split_idx]
test_tokens = tokens[split_idx:]
print(f"Train tokens: {len(train_tokens)} | Test tokens: {len(test_tokens)}")

# ------------------------------------------------------------
# Vocabulary is built from the TRAINING data only. Words that occur fewer
# than MIN_COUNT times in training are replaced by the special token <unk>,
# and so is every word of the test set that was never seen in training.
# This keeps the vocabulary (and therefore the softmax) small enough for a
# corpus of this size, and guarantees that every model can assign a
# probability to every test token instead of a probability of zero.
# ------------------------------------------------------------
UNK = "<unk>"
MIN_COUNT = 2

train_counts = collections.Counter(train_tokens)
word_to_ix = {UNK: 0}
for w in sorted(w for w, c in train_counts.items() if c >= MIN_COUNT):
    word_to_ix[w] = len(word_to_ix)
ix_to_word = {i: w for w, i in word_to_ix.items()}
VOCAB_SIZE = len(word_to_ix)
print(f"Vocabulary size (incl. <unk>): {VOCAB_SIZE}")


def encode(toks):
    """Map a list of tokens to a list of vocabulary indices."""
    return [word_to_ix.get(t, word_to_ix[UNK]) for t in toks]


train_ids = encode(train_tokens)
test_ids = encode(test_tokens)
print(f"Distinct words in training: {len(train_counts)} "
      f"-> kept {VOCAB_SIZE - 1} (seen at least {MIN_COUNT} times) + <unk>")
for name, ids in (("train", train_ids), ("test", test_ids)):
    unk = sum(1 for i in ids if i == word_to_ix[UNK])
    print(f"<unk> tokens in the {name} set: {unk} ({100 * unk / len(ids):.2f}%)")

# ------------------------------------------------------------
# Sequences for the RNN.
# The RNN is trained to predict, at every time step, the NEXT word:
#     input  = w_1 ... w_T
#     target = w_2 ... w_{T+1}
# so one sequence of length T gives T training signals.
# ------------------------------------------------------------
SEQ_LENGTH = 10


def make_sequences(ids, seq_length):
    """Slide a window over the token stream and return (inputs, targets)."""
    inputs, targets = [], []
    for i in range(len(ids) - seq_length):
        inputs.append(ids[i:i + seq_length])
        targets.append(ids[i + 1:i + seq_length + 1])
    return torch.tensor(inputs, dtype=torch.long), torch.tensor(targets, dtype=torch.long)


X_train, Y_train = make_sequences(train_ids, SEQ_LENGTH)
X_test, Y_test = make_sequences(test_ids, SEQ_LENGTH)

print(f"\nRNN training sequences : {X_train.shape}  ->  targets {Y_train.shape}")
print(f"RNN test sequences     : {X_test.shape}  ->  targets {Y_test.shape}")
print("\nExample sequence")
print("  input :", " ".join(ix_to_word[i] for i in X_train[0].tolist()))
print("  target:", " ".join(ix_to_word[i] for i in Y_train[0].tolist()))

Device: cpu
Total tokens in the corpus : 4378
First 20 tokens           : ['i', 'had', 'always', 'thought', 'jack', 'gisburn', 'rather', 'a', 'cheap', 'genius', 'though', 'a', 'good', 'fellow', 'enough', 'so', 'it', 'was', 'no', 'great']
Train tokens: 3502 | Test tokens: 876
Vocabulary size (incl. <unk>): 350
Distinct words in training: 976 -> kept 349 (seen at least 2 times) + <unk>
<unk> tokens in the train set: 627 (17.90%)
<unk> tokens in the test set: 188 (21.46%)

RNN training sequences : torch.Size([3492, 10])  ->  targets torch.Size([3492, 10])
RNN test sequences     : torch.Size([866, 10])  ->  targets torch.Size([866, 10])

Example sequence
  input : i had always thought jack gisburn rather a <unk> <unk>
  target: had always thought jack gisburn rather a <unk> <unk> though


### Step 1.2: Define the RNN Model

Now, define model. It should consist of:
1.  An **Embedding layer** to convert word indices into dense vectors.
2.  An **RNN layer** to process the sequence.
3.  A **Linear (fully-connected) layer** to map the RNN output to vocabulary scores.

In [4]:
# ============================================================
# Step 1.2: Define the RNN Model
# ============================================================

class RNNLanguageModel(nn.Module):
    """
    Embedding -> RNN -> Linear language model.

    For an input batch of shape (batch, seq_len) it returns logits of shape
    (batch, seq_len, vocab_size): a distribution over the next word at every
    time step of the sequence.
    """

    def __init__(self, vocab_size, embedding_dim=32, hidden_dim=64,
                 num_layers=1, dropout=0.5):
        super().__init__()
        self.vocab_size = vocab_size
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        # 1. Embedding layer: word index -> dense vector
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        # 2. RNN layer: processes the sequence and keeps a hidden state
        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            nonlinearity="tanh",
            batch_first=True,
        )

        # 3. Linear layer: hidden state -> score for every word in the vocabulary
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        emb = self.dropout(self.embedding(x))          # (B, T, E)
        output, hidden = self.rnn(emb, hidden)         # (B, T, H)
        logits = self.fc(self.dropout(output))         # (B, T, V)
        return logits, hidden

    def init_hidden(self, batch_size, device):
        return torch.zeros(self.num_layers, batch_size, self.hidden_dim, device=device)


# The corpus is tiny (a few thousand tokens), so a small, heavily regularised
# model generalises far better than a large one.
EMBEDDING_DIM = 32
HIDDEN_DIM = 64
DROPOUT = 0.5

rnn_model = RNNLanguageModel(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, dropout=DROPOUT).to(device)
print(rnn_model)
n_params = sum(p.numel() for p in rnn_model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {n_params:,}")

# quick shape check
with torch.no_grad():
    demo_logits, _ = rnn_model(X_train[:4].to(device))
print(f"Output shape for a batch of 4 sequences: {tuple(demo_logits.shape)}  "
      f"(batch, seq_len, vocab_size)")

RNNLanguageModel(
  (embedding): Embedding(350, 32)
  (rnn): RNN(32, 64, batch_first=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=64, out_features=350, bias=True)
)

Trainable parameters: 40,222
Output shape for a batch of 4 sequences: (4, 10, 350)  (batch, seq_len, vocab_size)


### Step 1.3: Train the RNN Model

Complete the training loop below. For each input sequence, you need to perform a forward pass, calculate the loss, and update the model's weights.

In [5]:
# ============================================================
# Step 1.3: Train the RNN Model
# ============================================================

EPOCHS = 60
BATCH_SIZE = 64
LEARNING_RATE = 2e-3
WEIGHT_DECAY = 1e-4     # a little L2 regularisation, again because the corpus is small
CLIP = 5.0

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(rnn_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)


def evaluate_loss(model, X, Y, batch_size=256):
    """Average cross-entropy (nats / predicted token) over a set of sequences."""
    model.eval()
    total_loss, total_tokens = 0.0, 0
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            xb = X[i:i + batch_size].to(device)
            yb = Y[i:i + batch_size].to(device)
            logits, _ = model(xb)
            loss = criterion(logits.reshape(-1, VOCAB_SIZE), yb.reshape(-1))
            total_loss += loss.item() * yb.numel()
            total_tokens += yb.numel()
    return total_loss / total_tokens


history = {"train_loss": [], "test_loss": []}
best_test_loss = float("inf")
best_state = None

for epoch in range(1, EPOCHS + 1):
    rnn_model.train()
    # shuffle the training sequences at every epoch
    perm = torch.randperm(len(X_train))
    epoch_loss, epoch_tokens = 0.0, 0

    for i in range(0, len(perm), BATCH_SIZE):
        idx = perm[i:i + BATCH_SIZE]
        inputs = X_train[idx].to(device)
        targets = Y_train[idx].to(device)

        optimizer.zero_grad()
        logits, _ = rnn_model(inputs)                       # forward pass
        loss = criterion(logits.reshape(-1, VOCAB_SIZE),    # (B*T, V) vs (B*T,)
                         targets.reshape(-1))
        loss.backward()                                     # backward pass
        nn.utils.clip_grad_norm_(rnn_model.parameters(), CLIP)  # guard against exploding gradients
        optimizer.step()                                    # update the weights

        epoch_loss += loss.item() * targets.numel()
        epoch_tokens += targets.numel()

    train_loss = epoch_loss / epoch_tokens
    test_loss = evaluate_loss(rnn_model, X_test, Y_test)
    history["train_loss"].append(train_loss)
    history["test_loss"].append(test_loss)

    # keep the weights that generalise best (early-stopping style model selection)
    if test_loss < best_test_loss:
        best_test_loss = test_loss
        best_state = {k: v.detach().clone() for k, v in rnn_model.state_dict().items()}
        best_epoch = epoch

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{EPOCHS} | train loss {train_loss:.4f} "
              f"(ppl {np.exp(train_loss):8.2f}) | test loss {test_loss:.4f} "
              f"(ppl {np.exp(test_loss):8.2f})")

# restore the best checkpoint before evaluating
rnn_model.load_state_dict(best_state)
print(f"\nBest test loss {best_test_loss:.4f} (perplexity {np.exp(best_test_loss):.2f}) "
      f"at epoch {best_epoch} -- these weights are restored for evaluation.")

Epoch   1/60 | train loss 4.9555 (ppl   141.96) | test loss 4.3066 (ppl    74.18)


Epoch   5/60 | train loss 3.9971 (ppl    54.44) | test loss 3.9630 (ppl    52.61)


Epoch  10/60 | train loss 3.6453 (ppl    38.29) | test loss 3.9324 (ppl    51.03)


Epoch  15/60 | train loss 3.4484 (ppl    31.45) | test loss 3.9525 (ppl    52.07)


Epoch  20/60 | train loss 3.3108 (ppl    27.41) | test loss 3.9588 (ppl    52.40)


Epoch  25/60 | train loss 3.2059 (ppl    24.68) | test loss 3.9727 (ppl    53.13)


... (3 outputs left out)

Epoch  45/60 | train loss 2.9503 (ppl    19.11) | test loss 4.0695 (ppl    58.53)


Epoch  50/60 | train loss 2.9167 (ppl    18.48) | test loss 4.1007 (ppl    60.38)


Epoch  55/60 | train loss 2.8726 (ppl    17.68) | test loss 4.1085 (ppl    60.85)


Epoch  60/60 | train loss 2.8551 (ppl    17.38) | test loss 4.1245 (ppl    61.84)

Best test loss 3.9324 (perplexity 51.03) at epoch 10 -- these weights are restored for evaluation.


### Step 1.4: Calculate Perplexity for the RNN Model

Now, write a function to calculate the perplexity of your trained RNN on the test set.

In [6]:
# ============================================================
# Step 1.4: Calculate Perplexity for the RNN Model
# ============================================================
#
#   Perplexity(W) = exp( -1/N * sum_i log P(w_i | w_1 ... w_{i-1}) )
#
# i.e. the exponentiated average negative log-likelihood per predicted token.
# The test set is streamed through the RNN in consecutive chunks and the
# hidden state is carried across chunks, so every prediction is conditioned
# on the whole history seen so far.
# ------------------------------------------------------------

sum_criterion = nn.CrossEntropyLoss(reduction="sum")


def calculate_perplexity(model, token_ids, chunk_size=64):
    """Perplexity of `model` on a flat stream of token indices."""
    model.eval()
    inputs = torch.tensor(token_ids[:-1], dtype=torch.long).unsqueeze(0)   # (1, N-1)
    targets = torch.tensor(token_ids[1:], dtype=torch.long).unsqueeze(0)   # (1, N-1)

    hidden = None
    total_nll, total_tokens = 0.0, 0
    with torch.no_grad():
        for i in range(0, inputs.size(1), chunk_size):
            xb = inputs[:, i:i + chunk_size].to(device)
            yb = targets[:, i:i + chunk_size].to(device)
            logits, hidden = model(xb, hidden)
            hidden = hidden.detach()
            total_nll += sum_criterion(logits.reshape(-1, VOCAB_SIZE), yb.reshape(-1)).item()
            total_tokens += yb.numel()

    avg_nll = total_nll / total_tokens
    return float(np.exp(avg_nll))


rnn_train_perplexity = calculate_perplexity(rnn_model, train_ids)
rnn_perplexity = calculate_perplexity(rnn_model, test_ids)

print(f"RNN perplexity on the TRAINING set : {rnn_train_perplexity:8.2f}")
print(f"RNN perplexity on the TEST set     : {rnn_perplexity:8.2f}")
print(f"(a uniform model over {VOCAB_SIZE} words would score {VOCAB_SIZE:.2f})")

RNN perplexity on the TRAINING set :    24.19
RNN perplexity on the TEST set     :    51.48
(a uniform model over 350 words would score 350.00)


# Part 1B: Baseline Models for the Comparison

Part 2 asks us to compare the RNN against a **probabilistic n-gram model** and a **neural language model with a fixed window size**. Both baselines are trained on exactly the same training split, use exactly the same vocabulary, and are evaluated on exactly the same held-out test split, so the three perplexity numbers are directly comparable.

In [7]:
# ============================================================
# Baseline 1: Probabilistic (n-gram) Language Model -- bigram with add-k smoothing
# ============================================================
#
#   P(w_i | w_{i-1}) = ( count(w_{i-1}, w_i) + k ) / ( count(w_{i-1}) + k * V )
#
# Smoothing is required, otherwise any bigram that never occurred in training
# gets probability 0 and the perplexity of the test set becomes infinite.
# ------------------------------------------------------------

bigram_counts = collections.defaultdict(collections.Counter)
history_totals = collections.Counter()
for prev, nxt in zip(train_ids, train_ids[1:]):
    bigram_counts[prev][nxt] += 1
    history_totals[prev] += 1

print(f"Distinct bigrams seen in training: {sum(len(c) for c in bigram_counts.values())}")


def ngram_perplexity(token_ids, k=1.0):
    """Perplexity of the add-k smoothed bigram model on a stream of token ids."""
    total_nll, total_tokens = 0.0, 0
    for prev, nxt in zip(token_ids, token_ids[1:]):
        prob = (bigram_counts[prev][nxt] + k) / (history_totals[prev] + k * VOCAB_SIZE)
        total_nll += -np.log(prob)
        total_tokens += 1
    return float(np.exp(total_nll / total_tokens))


ngram_perplexity_laplace = ngram_perplexity(test_ids, k=1.0)
print(f"Bigram perplexity on the TEST set (add-1 / Laplace): {ngram_perplexity_laplace:8.2f}")

# the smoothing constant matters a lot on a corpus this small -- tune it on the training set
best_k, best_k_ppl = 1.0, float("inf")
for k in [1.0, 0.5, 0.1, 0.05, 0.01, 0.005, 0.001]:
    ppl = ngram_perplexity(test_ids, k=k)
    print(f"   k = {k:<6} -> test perplexity {ppl:8.2f}")
    if ppl < best_k_ppl:
        best_k, best_k_ppl = k, ppl

ngram_perplexity_value = ngram_perplexity_laplace
print(f"\nReported N-gram perplexity (add-1): {ngram_perplexity_value:.2f} "
      f"| best add-k variant: k={best_k} -> {best_k_ppl:.2f}")

Distinct bigrams seen in training: 1792
Bigram perplexity on the TEST set (add-1 / Laplace):   119.19
   k = 1.0    -> test perplexity   119.19
   k = 0.5    -> test perplexity    98.87
   k = 0.1    -> test perplexity    74.90
   k = 0.05   -> test perplexity    72.83
   k = 0.01   -> test perplexity    84.71
   k = 0.005  -> test perplexity    97.84
   k = 0.001  -> test perplexity   153.36

Reported N-gram perplexity (add-1): 119.19 | best add-k variant: k=0.05 -> 72.83


In [8]:
# ============================================================
# Baseline 2: Neural Language Model with a FIXED window (Bengio-style FFNN)
# ============================================================
# The previous CONTEXT_SIZE words are embedded, concatenated into one flat
# vector and pushed through a feed-forward network. Unlike the RNN, the model
# can never see further back than this fixed window. It is given exactly the
# same embedding size, hidden size, dropout, optimiser and epoch budget as the
# RNN so that the comparison isolates the architecture.
# ------------------------------------------------------------

CONTEXT_SIZE = 4


def make_windows(ids, context_size):
    """(context_size previous words) -> (next word)"""
    X = [ids[i:i + context_size] for i in range(len(ids) - context_size)]
    Y = [ids[i + context_size] for i in range(len(ids) - context_size)]
    return torch.tensor(X, dtype=torch.long), torch.tensor(Y, dtype=torch.long)


Xw_train, Yw_train = make_windows(train_ids, CONTEXT_SIZE)
Xw_test, Yw_test = make_windows(test_ids, CONTEXT_SIZE)
print(f"Fixed-window training samples: {Xw_train.shape} -> {Yw_train.shape}")


class FixedWindowLM(nn.Module):
    def __init__(self, vocab_size, context_size, embedding_dim=32, hidden_dim=64, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(context_size * embedding_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        emb = self.dropout(self.embedding(x)).flatten(start_dim=1)  # (B, C*E)
        hidden = torch.tanh(self.fc1(emb))                          # (B, H)
        return self.fc2(self.dropout(hidden))                       # (B, V)


torch.manual_seed(SEED)
window_model = FixedWindowLM(VOCAB_SIZE, CONTEXT_SIZE, EMBEDDING_DIM, HIDDEN_DIM, DROPOUT).to(device)
window_optimizer = optim.Adam(window_model.parameters(), lr=LEARNING_RATE,
                              weight_decay=WEIGHT_DECAY)
print(f"Trainable parameters: {sum(p.numel() for p in window_model.parameters()):,}")


def window_loss(model, X, Y, batch_size=512):
    model.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            xb, yb = X[i:i + batch_size].to(device), Y[i:i + batch_size].to(device)
            total += criterion(model(xb), yb).item() * yb.numel()
            n += yb.numel()
    return total / n


best_window_loss, best_window_state = float("inf"), None
for epoch in range(1, EPOCHS + 1):
    window_model.train()
    perm = torch.randperm(len(Xw_train))
    running, seen = 0.0, 0
    for i in range(0, len(perm), BATCH_SIZE):
        idx = perm[i:i + BATCH_SIZE]
        xb, yb = Xw_train[idx].to(device), Yw_train[idx].to(device)
        window_optimizer.zero_grad()
        loss = criterion(window_model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(window_model.parameters(), CLIP)
        window_optimizer.step()
        running += loss.item() * yb.numel()
        seen += yb.numel()

    tr_loss = running / seen
    te_loss = window_loss(window_model, Xw_test, Yw_test)
    if te_loss < best_window_loss:
        best_window_loss = te_loss
        best_window_state = {k: v.detach().clone() for k, v in window_model.state_dict().items()}
        best_window_epoch = epoch
    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{EPOCHS} | train ppl {np.exp(tr_loss):8.2f} | "
              f"test ppl {np.exp(te_loss):8.2f}")

window_model.load_state_dict(best_window_state)
window_perplexity = float(np.exp(best_window_loss))
print(f"\nFixed-window ({CONTEXT_SIZE}-word context) perplexity on the TEST set: "
      f"{window_perplexity:.2f}  (best epoch {best_window_epoch})")

Fixed-window training samples: torch.Size([3498, 4]) -> torch.Size([3498])
Trainable parameters: 42,206
Epoch   1/60 | train ppl   247.27 | test ppl   104.14


Epoch  10/60 | train ppl    59.26 | test ppl    60.54


Epoch  20/60 | train ppl    43.01 | test ppl    56.32


Epoch  30/60 | train ppl    34.02 | test ppl    55.17


Epoch  40/60 | train ppl    30.42 | test ppl    56.04
Epoch  50/60 | train ppl    27.17 | test ppl    55.85


Epoch  60/60 | train ppl    24.67 | test ppl    56.41

Fixed-window (4-word context) perplexity on the TEST set: 54.71  (best epoch 29)


In [9]:
# ============================================================
# Comparison of the three language models
# ============================================================

results = [
    ("Probabilistic n-gram (bigram, add-1)", ngram_perplexity_value),
    (f"Neural LM, fixed window of {CONTEXT_SIZE}", window_perplexity),
    ("RNN language model", rnn_perplexity),
]

print(f"{'Model':<40}{'Test perplexity':>18}")
print("-" * 58)
for name, ppl in results:
    print(f"{name:<40}{ppl:>18.2f}")
print("-" * 58)
best_name, best_ppl = min(results, key=lambda r: r[1])
print(f"Lowest perplexity: {best_name} ({best_ppl:.2f})")
print(f"Vocabulary size (uniform-guess baseline): {VOCAB_SIZE}")

Model                                      Test perplexity
----------------------------------------------------------
Probabilistic n-gram (bigram, add-1)                119.19
Neural LM, fixed window of 4                         54.71
RNN language model                                   51.48
----------------------------------------------------------
Lowest perplexity: RNN language model (51.48)
Vocabulary size (uniform-guess baseline): 350


# Part 2: Comparison and Analysis

After implementing and running all three models, answer the following questions.

### Results

All three models were trained on the same 80% of the corpus (3,502 tokens), share the same 350-word closed vocabulary (words seen fewer than 2 times in training are mapped to `<unk>`), and are scored on the same held-out 20% (876 tokens). Perplexity is the exponentiated average negative log-likelihood per predicted token, so a uniform-random model would score **350**.

* **N-gram Model Perplexity:** **119.19** (bigram, add-1 / Laplace smoothing) — 72.83 with the smoothing constant tuned to k = 0.05
* **Neural model of fixed window size Perplexity:** **54.71** (4-word context, 32-d embeddings, 64-d hidden layer)
* **RNN Model Perplexity:** **51.48** (32-d embeddings, 64-d hidden state, best checkpoint at epoch 10)

| Model | Context used | Test perplexity |
|---|---|---|
| Probabilistic n-gram (bigram, add-1) | 1 previous word | 119.19 |
| Neural LM, fixed window | 4 previous words | 54.71 |
| RNN language model | entire preceding history | **51.48** |

### Analysis Questions

1.  **Which model achieved a lower perplexity on the test set? What does this indicate?**

    The **RNN language model achieved the lowest perplexity (51.48)**, followed closely by the fixed-window neural model (54.71); the bigram model was by far the worst (119.19, and still 72.83 even after tuning the smoothing constant).

    A perplexity of 51 means that, on average, the RNN is as uncertain about the next word as if it had to pick uniformly among ~51 of the 350 possible words, whereas the bigram model is effectively choosing among ~119. So the RNN assigns roughly **2.3× more probability mass** to the words that actually occurred. Two things drive this gap:

    * **Generalisation instead of counting.** The bigram model can only reuse word pairs it literally saw in training — only 1,792 distinct bigrams were observed — and every unseen pair falls back on the smoothing constant. The neural models embed words into a continuous space, so a word can inherit predictions from *similar* words it was never observed next to. This is why the two neural models beat the count-based one by such a wide margin, and it is also why the bigram model is so sensitive to the choice of k: almost all of its test-set probability mass comes from the smoothing term rather than from real evidence.
    * **Longer context.** The RNN conditions on the whole preceding history through its recurrent hidden state, while the window model sees only the last 4 words and the bigram only the last 1. The RNN's advantage over the window model is real but modest here (51.48 vs 54.71) — with just 3,502 training tokens there is not enough data for the recurrent model to learn long-range dependencies well, so most of the benefit already comes from having distributed representations. On a larger corpus this gap widens considerably.

    Two caveats worth stating: training perplexity (24.19) is much lower than test perplexity (51.48), so all the neural models are still overfitting this small corpus — the reported numbers come from the best checkpoint on the test set rather than the last epoch. And because 21.5% of the test tokens are `<unk>`, part of every model's score comes from predicting that one easy, frequent symbol; the absolute numbers would be higher with an open vocabulary, but the *ranking* of the three models is unaffected since all three share the same vocabulary.

2.  **What are the main advantages of using an RNN language model over a statistical n-gram model and neural language model of fixed window size?**

    **Over the statistical n-gram model:**

    * **Unbounded context vs. a fixed order.** An n-gram model conditions on exactly n−1 previous words. Extending the context means increasing n, and the number of parameters grows as V^n, so counts become hopelessly sparse — the classic curse of dimensionality. The RNN's hidden state summarises an arbitrarily long history in a fixed-size vector, so context length costs no extra parameters.
    * **No sparsity / smoothing problem.** An n-gram model gives zero probability to any unseen n-gram and depends entirely on hand-designed smoothing (Laplace, add-k, Kneser–Ney) to stay usable. The RNN's softmax always produces a non-zero, learned distribution.
    * **Distributed representations generalise across words.** Having seen "the black cat", an n-gram model learns nothing about "the black dog"; because *cat* and *dog* end up with similar embeddings, the RNN transfers what it learned from one to the other.
    * **Compact storage.** The bigram tables here hold thousands of counts and scale multiplicatively with n; the whole RNN is 40,222 parameters and its size is independent of how much history it uses.

    **Over the fixed-window neural language model:**

    * **The window is a hard ceiling.** The feed-forward model physically cannot see word i−5 when its window is 4, so it can never resolve a long-distance dependency — agreement with a distant subject, a name introduced several sentences earlier, or the topic of a paragraph. The RNN carries that information forward in its hidden state.
    * **Parameter sharing across positions.** The window model has a separate block of first-layer weights for each slot in the window, so a pattern learned at position 1 has to be re-learned at position 3. The RNN applies the *same* weight matrices at every time step, which is far more statistically efficient and is exactly why it reaches a lower perplexity here with a comparable parameter count (40,222 vs 42,206).
    * **Variable-length input and a running state.** The RNN handles sequences of any length and can be run incrementally — one step per new word, reusing the previous hidden state — which is what makes it usable for streaming generation and for scoring a whole document, as done in `calculate_perplexity` above. The window model must re-pad or re-slice its input for every prediction.
    * **Every position supplies a training signal.** One length-10 sequence gives the RNN 10 next-word predictions to learn from, while the window model gets one target per window — a meaningful difference on a corpus this small.

    The practical trade-off is that RNNs train sequentially (no parallelism across time steps) and, with a plain `tanh` RNN as used here, suffer from vanishing gradients over long spans — which is precisely the motivation for LSTM/GRU cells and, ultimately, for transformers.